In [ ]:
import pandas as pd
import numpy as np

train_ratings = pd.read_csv(
    "../datasets/processed/train_ratings.csv"
)

test_ratings = pd.read_csv(
    "../datasets/processed/test_ratings.csv"
)

recipes = pd.read_csv(
    "../../datasets/RAW_recipes.csv"
)

print("Train shape:", train_ratings.shape)
print("Test shape:", test_ratings.shape)
print("Recipes shape:", recipes.shape)

In [ ]:
import os

print(os.getcwd())

In [ ]:
recipe_popularity = (
    train_ratings
    .groupby("recipe_id")
    .size()
    .reset_index(name="rating_count")
)

recipe_popularity = recipe_popularity.sort_values(
    "rating_count",
    ascending=False
)

recipe_popularity.head(10)

In [ ]:
top_popular_recipes = recipe_popularity.merge(
    recipes[["id", "name"]],
    left_on="recipe_id",
    right_on="id",
    how="left"
)

top_popular_recipes = top_popular_recipes[
    ["recipe_id", "name", "rating_count"]
].head(10)

top_popular_recipes

In [ ]:
def recommend_popular(user_id, n=10):
    """
    Recommend the N most popular recipes
    that the user has not already rated.
    """

    user_rated = set(
        train_ratings.loc[
            train_ratings["user_id"] == user_id,
            "recipe_id"
        ]
    )

    recommendations = recipe_popularity[
        ~recipe_popularity["recipe_id"].isin(user_rated)
    ].head(n)

    return recommendations

In [ ]:
test_user_id = test_ratings["user_id"].iloc[0]

print("Test user:", test_user_id)

recommendations = recommend_popular(
    test_user_id,
    n=10
)

recommendations

In [ ]:
def precision_at_k(user_id, recommendations, k=10):
    """
    Calculate Precision@K for one user.
    """

    actual = set(
        test_ratings.loc[
            test_ratings["user_id"] == user_id,
            "recipe_id"
        ]
    )

    recommended = set(
        recommendations.head(k)["recipe_id"]
    )

    hits = len(actual.intersection(recommended))

    return hits / k

In [ ]:
precision = precision_at_k(
    test_user_id,
    recommendations,
    k=10
)

print("Precision@10:", precision)

In [ ]:
def evaluate_popularity_model(k=10):
    precisions = []

    test_users = test_ratings["user_id"].unique()

    for user_id in test_users:
        recommendations = recommend_popular(user_id, n=k)

        precision = precision_at_k(
            user_id,
            recommendations,
            k=k
        )

        precisions.append(precision)

    return np.mean(precisions)

In [ ]:
precision_at_10 = evaluate_popularity_model(k=10)

print("Popularity Baseline Precision@10:", precision_at_10)

In [ ]:
def recall_at_k(user_id, recommendations, k=10):
    """
    Calculate Recall@K for one user.
    """

    actual = set(
        test_ratings.loc[
            test_ratings["user_id"] == user_id,
            "recipe_id"
        ]
    )

    recommended = set(
        recommendations.head(k)["recipe_id"]
    )

    if len(actual) == 0:
        return 0.0

    hits = len(actual.intersection(recommended))

    return hits / len(actual)

In [ ]:
def evaluate_recall_popularity_model(k=10):
    recalls = []

    test_users = test_ratings["user_id"].unique()

    for user_id in test_users:
        recommendations = recommend_popular(user_id, n=k)

        recall = recall_at_k(
            user_id,
            recommendations,
            k=k
        )

        recalls.append(recall)

    return np.mean(recalls)

In [ ]:
recall_at_10 = evaluate_recall_popularity_model(k=10)

print("Popularity Baseline Recall@10:", recall_at_10)

In [ ]:
import pickle

popularity_model = {
    "recipe_popularity": recipe_popularity,
    "k": 10,
    "precision_at_10": precision_at_10,
    "recall_at_10": recall_at_10
}
with open("../saved_models/popularity_baseline.pkl", "wb") as f:
    pickle.dump(popularity_model, f)

print("Popularity Baseline model saved.")

In [ ]:


with open("../saved_models/popularity_baseline.pkl", "rb") as f:
    loaded_model = pickle.load(f)

print("Model successfully loaded.")
print("Precision@10:", loaded_model["precision_at_10"])
print("Recall@10:", loaded_model["recall_at_10"])

In [ ]:
def recommend_from_saved_model(user_id, n=10):
    """
    Generate recommendations using the saved Popularity Baseline model.
    """

    popularity = loaded_model["recipe_popularity"]

    user_rated = set(
        train_ratings.loc[
            train_ratings["user_id"] == user_id,
            "recipe_id"
        ]
    )

    recommendations = popularity[
        ~popularity["recipe_id"].isin(user_rated)
    ].head(n)

    return recommendations

In [ ]:
saved_recommendations = recommend_from_saved_model(
    user_id=162826,
    n=10
)

saved_recommendations